# M3L4 E06 — Evaluator Agent simple
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

**Objetivo:** crear un evaluador automático que puntúa la calidad de las respuestas de los agentes.

## ¿Por qué evaluar respuestas?

Medir el routing accuracy no es suficiente. También necesitamos saber:
- ¿La respuesta contiene la información correcta?
- ¿Es clara y completa?
- ¿No alucina datos que no existen?

## Dos enfoques

| Evaluador | Ventaja | Limitación |
|---|---|---|
| **Determinístico** (keywords) | Rápido, reproducible, sin LLM | Limitado a vocabulario fijo |
| **LLM-as-judge** | Entiende semántica, contexto | Costo, latencia, necesita calibración |

## Parte 1 — Evaluador determinístico por keywords

In [ ]:
def simple_keyword_evaluator(expected_keywords: list, actual_answer: str) -> dict:
    """
    Evalúa si la respuesta del agente contiene las keywords esperadas.

    Args:
        expected_keywords: lista de palabras que deben estar en la respuesta
        actual_answer: respuesta generada por el agente

    Returns:
        dict con score (0.0–1.0), matched_keywords y reason
    """
    # TODO 1: manejar el caso donde expected_keywords está vacía
    # devolver score 0, reason 'No expected keywords provided.'

    # TODO 2: verificar cuáles keywords están en actual_answer (case insensitive)
    matched = []

    # TODO 3: calcular score = matched / total keywords (redondeado a 2 decimales)
    score = 0

    # TODO 4: retornar dict con score, matched_keywords y reason
    return {}

print('Función definida.')

## Casos de prueba

In [ ]:
# Caso 1: respuesta perfecta
result1 = simple_keyword_evaluator(
    expected_keywords=['factura', 'pagos', 'portal'],
    actual_answer='Podés ver tu factura desde el portal de pagos.'
)
print('Caso 1 (perfecto):', result1)

In [ ]:
# Caso 2: respuesta parcial
result2 = simple_keyword_evaluator(
    expected_keywords=['vacaciones', 'solicitud', 'formulario', 'portal'],
    actual_answer='Para pedir vacaciones completá el formulario.'
)
print('Caso 2 (parcial):', result2)

In [ ]:
# Caso 3: respuesta incorrecta
result3 = simple_keyword_evaluator(
    expected_keywords=['factura', 'pagos', 'portal'],
    actual_answer='Probá reiniciar la app.'
)
print('Caso 3 (incorrecto):', result3)

In [ ]:
# Caso 4: sin keywords esperadas
result4 = simple_keyword_evaluator(
    expected_keywords=[],
    actual_answer='Probá reiniciar la app.'
)
print('Caso 4 (sin keywords):', result4)

## Parte 2 — Evaluar el golden dataset completo

In [ ]:
import pandas as pd

# Respuestas simuladas de los agentes (algunas correctas, algunas no)
evaluation_cases = [
    {
        'query': 'Necesito ver mi factura del mes pasado',
        'expected_keywords': ['factura', 'portal', 'pagos'],
        'agent_response': 'Podés ver tu factura desde el portal de pagos.'
    },
    {
        'query': '¿Cómo solicito mis días de vacaciones?',
        'expected_keywords': ['vacaciones', 'portal', 'formulario'],
        'agent_response': 'Para solicitar vacaciones ingresá al portal de RRHH y completá el formulario.'
    },
    {
        'query': 'Mi VPN no conecta desde ayer',
        'expected_keywords': ['VPN', 'conexión', 'soporte'],
        'agent_response': 'Probá reiniciar el router.'
    },
    {
        'query': '¿Cuándo se procesa el reembolso de gastos?',
        'expected_keywords': ['reembolso', 'gastos', '48 horas'],
        'agent_response': 'Los gastos se procesan dentro de las 48 horas hábiles.'
    },
    {
        'query': 'Necesito el contrato de confidencialidad',
        'expected_keywords': ['contrato', 'confidencialidad', 'legal'],
        'agent_response': 'Completá el formulario de RRHH para acceder al contrato.'
    },
]

print(f'Casos de evaluación: {len(evaluation_cases)}')

In [ ]:
# TODO: evaluar cada caso y mostrar tabla de resultados
results = []
for case in evaluation_cases:
    eval_result = simple_keyword_evaluator(
        case['expected_keywords'],
        case['agent_response']
    )
    results.append({
        'query': case['query'][:40],
        'score': eval_result.get('score'),
        'matched': eval_result.get('matched_keywords'),
        'reason': eval_result.get('reason')
    })

df = pd.DataFrame(results)
print(f'Quality score promedio: {df["score"].mean():.2f}')
df

## Parte 3 — Esqueleto del evaluador LLM-as-judge (extensión)

Este código es **conceptual** — requiere un LLM conectado. Lo verás en producción en E19 de M3L3 y E08+ de este módulo.

In [ ]:
def llm_evaluator_prompt(query: str, actual_answer: str) -> str:
    """Genera el prompt para un evaluador LLM."""
    return f"""Evaluá la siguiente respuesta de un agente de soporte interno.

Query del usuario:
{query}

Respuesta del agente:
{actual_answer}

Criterios de evaluación:
- Corrección factual (¿la información es correcta?)
- Completitud (¿responde la pregunta completa?)
- Claridad (¿es fácil de entender?)
- No alucinación (¿no inventa datos?)

Respondé SOLO con JSON sin markdown:
{{"score": 1-10, "reason": "explicación breve en una oración"}}
"""

# Mostrar el prompt para el primer caso
print(llm_evaluator_prompt(
    '¿Cómo solicito mis días de vacaciones?',
    'Para solicitar vacaciones ingresá al portal de RRHH y completá el formulario.'
))

In [ ]:
r = simple_keyword_evaluator(['factura', 'pagos', 'portal'], 'Podés ver tu factura desde el portal de pagos.')
assert r['score'] == 1.0, f'Score esperado 1.0, obtenido {r["score"]}'
assert len(r['matched_keywords']) == 3

r2 = simple_keyword_evaluator([], 'cualquier cosa')
assert r2['score'] == 0

print('Checks E06 OK ✅')